## Wikipedia Abstract Fallback Strategy

In [11]:

import httpx
import json

async def test_wikipedia_abstract_lookup(title: str = "Grundlegung zur Metaphysik der Sitten", author: str = "Immanuel Kant"):
    """
    Demonstrates how we fall back to Wikipedia. We search OpenSearch using 
    just the book title, since Wikipedia article titles match the book name.
    """
    
    # Use just the title for OpenSearch title matching
    wiki_query = title.strip()
    print(f"[Wikipedia Test] Target title query: '{wiki_query}'")
    
    async with httpx.AsyncClient() as client:
        try:
            search_url = "https://de.wikipedia.org/w/api.php"
            search_params = {
                "action": "opensearch",
                "search": wiki_query,
                "limit": 1,
                "namespace": 0,
                "format": "json"
            }
            
            headers = {
                "User-Agent": "BibliographicEnrichmentAPI/1.0 (https://github.com/klausbprinz/bibinfo-enrichapi; klaus_prinz@gmx.net)"
            }
            
            print("[Wikipedia Test] Querying German Wikipedia OpenSearch API...")
            search_res = await client.get(search_url, params=search_params, headers=headers, timeout=3.0)
            
            if search_res.status_code != 200:
                print(f"[Wikipedia Test] Search failed with HTTP {search_res.status_code}")
                return None
                
            json_res = search_res.json()
            titles = json_res[1] if len(json_res) > 1 else []
            
            if not titles:
                print("[Wikipedia Test] No matching Wikipedia article found for this query.")
                return None
                
            page_title = titles[0]
            print(f"[Wikipedia Test] Success! Found matching Wikipedia page title: '{page_title}'")
            
            # Fetch summary extract using the resolved page title on German Wikipedia
            summary_url = f"https://de.wikipedia.org/api/rest_v1/page/summary/{page_title}"
            
            print(f"[Wikipedia Test] Fetching summary extract for '{page_title}'...")
            summary_res = await client.get(summary_url, headers=headers, timeout=3.0)
            
            if summary_res.status_code == 200:
                summary_data = summary_res.json()
                extract_text = summary_data.get("extract")
                
                print("\n--- Retrieved Wikipedia Abstract ---")
                print(extract_text)
                
                return {
                    "pageTitle": page_title,
                    "abstract": extract_text,
                    "sourceUrl": summary_data.get("content_urls", {}).get("desktop", {}).get("page")
                }
            else:
                print(f"[Wikipedia Test] Summary endpoint returned HTTP {summary_res.status_code}")
                
        except Exception as e:
            print(f"[Wikipedia Test] Request encountered an error: {e}")

# Run it:
# await test_wikipedia_abstract_lookup()

In [12]:
await test_wikipedia_abstract_lookup()

[Wikipedia Test] Target title query: 'Grundlegung zur Metaphysik der Sitten'
[Wikipedia Test] Querying German Wikipedia OpenSearch API...
[Wikipedia Test] Success! Found matching Wikipedia page title: 'Grundlegung zur Metaphysik der Sitten'
[Wikipedia Test] Fetching summary extract for 'Grundlegung zur Metaphysik der Sitten'...

--- Retrieved Wikipedia Abstract ---
Die Grundlegung zur Metaphysik der Sitten ist ein Werk von Immanuel Kant, das im Jahr 1785 im Verlag J. F. Hartknoch erschien. Es ist die erste grundlegende Schrift Kants zur Ethik, die er im schon recht hohen Alter von 61 Jahren veröffentlichte, nachdem er zuvor seine theoretische Philosophie mit der Kritik der reinen Vernunft formuliert hatte. In der GMS verfolgt Kant das Ziel, die Begründung einer Moralphilosophie zu entwerfen, die allein auf Überlegungen der reinen Vernunft beruht und deren Prinzipien weder aus einem metaphysischen Weltbild noch aus der von zufälligen Einflüssen bestimmten Erfahrung abgeleitet sind.


{'pageTitle': 'Grundlegung zur Metaphysik der Sitten',
 'abstract': 'Die Grundlegung zur Metaphysik der Sitten ist ein Werk von Immanuel Kant, das im Jahr 1785 im Verlag J. F. Hartknoch erschien. Es ist die erste grundlegende Schrift Kants zur Ethik, die er im schon recht hohen Alter von 61 Jahren veröffentlichte, nachdem er zuvor seine theoretische Philosophie mit der Kritik der reinen Vernunft formuliert hatte. In der GMS verfolgt Kant das Ziel, die Begründung einer Moralphilosophie zu entwerfen, die allein auf Überlegungen der reinen Vernunft beruht und deren Prinzipien weder aus einem metaphysischen Weltbild noch aus der von zufälligen Einflüssen bestimmten Erfahrung abgeleitet sind.',
 'sourceUrl': 'https://de.wikipedia.org/wiki/Grundlegung_zur_Metaphysik_der_Sitten'}